In [0]:
import requests 
import pandas as pd
import time
import json
from datetime import date,datetime

In [0]:
#Declare file config
file_path = '../credentials/config.json'

#Open and read Json file
with open(file_path, "r") as file:
    config_data = json.load(file)

In [0]:
current_date = date.today()
current_date = current_date.strftime('%d/%m/%Y')
print("Current Date:", current_date)


In [0]:
url = "https://fc-data.ssi.com.vn/api/v2/Market/AccessToken"

payload = {
    "consumerID": config_data["consumerID"],
    "consumerSecret": config_data["consumerSecret"]
}

headers = {
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)
if response.status_code == 200:
    access_token = response.json()['data'].get("accessToken")
else:
    print("Failed to get access token. Status Code:", response.status_code)
    print("Response Body:", response.text)
    exit()

In [0]:
url = "https://fc-data.ssi.com.vn/api/v2/Market/SecuritiesDetails"
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

df = pd.DataFrame()

for index in range(1, 10):
    params = {
        "pageIndex": index,
        "pageSize": 1000
    }
    response = requests.get(url, headers=headers, params=params)
    json_data = response.json()
    if (
        'data' not in json_data or
        json_data['data'] is None or
        len(json_data['data']) == 0 or
        'RepeatedInfo' not in json_data['data'][0] or
        json_data['data'][0]['RepeatedInfo'] is None
    ):
        break
    data = json_data['data'][0]['RepeatedInfo']
    df_response = pd.DataFrame(data)
    df = pd.concat([df, df_response], ignore_index=True)
    time.sleep(1)
    if len(df_response) == 0:
        break

In [0]:
display(df)

In [0]:
# Ensure all required columns are present and have correct types
required_columns = [
    "is_in", "symbol", "symbol_name", "symbol_eng_name", "sec_type", "market_id", "exchange", "issuer",
    "lot_size", "issue_date", "maturity_date", "first_trading_date", "last_trading_date",
    "contract_multiplier", "settl_method", "underlying", "put_or_call", "exercise_price",
    "exercise_style", "excercise_ratio", "listed_share", "tick_price_1", "tick_increment_1",
    "tick_price_2", "tick_increment_2", "tick_price_3", "tick_increment_3", "tick_price_4", "tick_increment_4"
]

# Update your schema mapping to match the table exactly
schema_mapping = {
    "Isin": "is_in",
    "Symbol": "symbol",
    "SymbolName": "symbol_name",
    "SymbolEngName": "symbol_eng_name",
    "SecType": "sec_type",
    "MarketID": "market_id",  # Make sure your source column is 'MarketID', not 'MarketId'
    "Exchange": "exchange",
    "Issuer": "issuer",
    "LotSize": "lot_size",
    "IssueDate": "issue_date",
    "MaturityDate": "maturity_date",
    "FirstTradingDate": "first_trading_date",
    "LastTradingDate": "last_trading_date",
    "ContractMultiplier": "contract_multiplier",
    "SettlMethod": "settl_method",
    "Underlying": "underlying",
    "PutOrCall": "put_or_call",
    "ExercisePrice": "exercise_price",
    "ExerciseStyle": "exercise_style",
    "ExcerciseRatio": "excercise_ratio",
    "ListedShare": "listed_share",
    "TickPrice1": "tick_price_1",
    "TickIncrement1": "tick_increment_1",
    "TickPrice2": "tick_price_2",
    "TickIncrement2": "tick_increment_2",
    "TickPrice3": "tick_price_3",
    "TickIncrement3": "tick_increment_3",
    "TickPrice4": "tick_price_4",
    "TickIncrement4": "tick_increment_4"
}

# Rename columns
df = df.rename(columns=schema_mapping)

# Add missing columns with null values and ensure correct order
for col in required_columns:
    if col not in df.columns:
        df[col] = None
df = df[required_columns]

# Convert all columns to string type to match the Delta table schema
df = df.astype(str)

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Write the Spark DataFrame to the table
spark_df.write.format("delta").mode("overwrite").saveAsTable("stock_data.ingestion_data.securities_details")